# Qwen3-4B — QLoRA Fine-tuning for Event Extraction

Notebook huấn luyện Qwen3-4B trên dữ liệu Event Extraction theo định dạng `input_text` → `target_text`.

**Các phần có thể tùy chỉnh khi chạy:**
- `model_path`: đường dẫn model nền.
- Đường dẫn `train.jsonl` và `dev.jsonl`: dataset.
- `CHECKPOINT` và `RESUME_FROM_CHECKPOINT`: chọn train từ đầu hoặc tiếp tục từ checkpoint.

> Các hyperparameter huấn luyện, cấu hình QLoRA và cách tokenize/collate được giữ nguyên theo code gốc.

# Link resourse Kaggle (model, dataset, notebook)
## Model
- Qwen3-4B: https://www.kaggle.com/datasets/md0612/qwen3-4b

## Dataset
- GENEVA: https://www.kaggle.com/datasets/md0612/geneva-adapted
- VHE: https://www.kaggle.com/datasets/md0612/vhe-adapted
- BKEE: https://www.kaggle.com/datasets/md0612/bkee-adapted
- PHEE: 

## Metrics
- https://www.kaggle.com/datasets/md0612/metrics

## Sample notebook này
- 

## 1. Cài đặt thư viện

Cài hoặc nâng cấp `bitsandbytes` để hỗ trợ tải model ở chế độ 4-bit.

In [ ]:
!pip install -U "bitsandbytes>=0.46.1"

## 2. Import thư viện

Import các thư viện dùng cho load model, xử lý dataset, QLoRA và training.

In [ ]:
from pathlib import Path
import json
import gc
import sys
import numpy as np

import torch
from datasets import load_dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
)
from peft import (
    prepare_model_for_kbit_training,
    get_peft_model,
    LoraConfig,
    PeftModel,
)

## 3. Load tokenizer và base model

Thiết lập đường dẫn model nền, cấu hình quantization 4-bit và load Qwen3-4B.

In [ ]:
# model_path = Path("## Model path")
model_path = Path("/kaggle/input/datasets/md0612/qwen3-4b")

# 4-bit quantization
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, # Load 4-bit, giảm VRAM
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(model_path) # Load tokenizer

# Load model
model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto",
)

# Chuẩn bị model cho QLoRA
model = prepare_model_for_kbit_training(model)

# Quan trọng khi dùng gradient checkpointing
model.config.use_cache = False
model.gradient_checkpointing_enable()

print("Model loaded!")
print("Device:", model.device)

## 4. Tokenize một example

Ghép token của input và target; chỉ tính loss trên phần target và EOS.

In [ ]:
def tokenize_example(example):
    input_text = example["input_text"]
    target_text = example["target_text"]

    input_tokens = tokenizer(
        input_text,
        add_special_tokens=True,
    )

    target_tokens = tokenizer(
        target_text,
        add_special_tokens=False,
    )

    input_ids = input_tokens["input_ids"]
    target_ids = target_tokens["input_ids"]

    input_ids = (
        input_tokens["input_ids"]
        + target_ids
        + [tokenizer.eos_token_id]
    ) # Thêm EOS để model dừng lại

    labels = (
        [-100] * len(input_tokens["input_ids"])
        + target_ids
        + [tokenizer.eos_token_id]
    ) # Thêm EOS để model dừng lại khi generate

    return {
        "input_ids": input_ids,
        "labels": labels,
        "attention_mask": [1] * len(input_ids),
    }

## 5. Format dữ liệu Event Extraction

Chuyển mỗi sample từ `text` và `events` sang cặp `input_text` / `target_text`.

In [ ]:
def format_example(example):
    text = example["text"]

    target = {
        "events": example["events"]
    }

    return {
        "input_text": text,
        "target_text": json.dumps(
            target,
            ensure_ascii=False,
            separators=(",", ":")
        )
    }

## 6. Data collator

Pad các sample trong batch; phần padding của `labels` được gán `-100` để không tính loss.

In [ ]:
def ee_data_collator(features):
    max_length = max(len(f["input_ids"]) for f in features)

    input_ids = []
    attention_mask = []
    labels = []

    for f in features:
        pad_length = max_length - len(f["input_ids"])

        input_ids.append(
            f["input_ids"] + [tokenizer.pad_token_id] * pad_length
        )

        attention_mask.append(
            f["attention_mask"] + [0] * pad_length
        )

        labels.append(
            f["labels"] + [-100] * pad_length
        )

    return {
        "input_ids": torch.tensor(input_ids, dtype=torch.long),
        "attention_mask": torch.tensor(attention_mask, dtype=torch.long),
        "labels": torch.tensor(labels, dtype=torch.long),
    }

## 7. Cấu hình và áp dụng LoRA

Khai báo cấu hình LoRA và gắn adapter vào model đã được chuẩn bị cho k-bit training.

In [ ]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(model, peft_config)

## 8. Load và preprocess dataset

Đọc train/dev JSONL, format dữ liệu rồi tokenize. Thay hai đường dẫn nếu muốn dùng dataset khác.

In [ ]:
# dataset = load_dataset(
#     "json",
#     data_files={
#         "train": "## Data path/train.jsonl",
#         "dev": "## Data path/dev.jsonl",
#     },
# )

dataset = load_dataset(
    "json",
    data_files={
        "train": "/kaggle/input/datasets/md0612/vhe-adapted/train.jsonl",
        "dev": "/kaggle/input/datasets/md0612/vhe-adapted/dev.jsonl",
    },
)

# Raw datasets
train_raw = dataset["train"]
dev_raw = dataset["dev"]

# Formatted datasets
train_formatted = train_raw.map(format_example)
dev_formatted = dev_raw.map(format_example)

# Tokenized datasets for training
train_dataset = train_formatted.map(
    tokenize_example,
    remove_columns=train_formatted.column_names,
)

# Tokenized dataset for validation loss (nếu sử dụng)
dev_dataset = dev_formatted.map(
    tokenize_example,
    remove_columns=dev_formatted.column_names,
)

# Raw/formatted dataset for generation evaluation
dev_generation_dataset = dev_formatted

## 9. (Tùy chọn) Metrics và early stopping

Phần evaluator/metrics dưới đây được giữ ở dạng comment như notebook gốc để tiện bật lại khi cần.

Khi muốn dùng early stopping trong tương lai, cần bổ sung `EarlyStoppingCallback` vào `Trainer`, bật evaluation trong `TrainingArguments` và cấu hình metric phù hợp. Các phần đó hiện không được kích hoạt để giữ nguyên thiết lập training gốc.

In [ ]:
# sys.path.insert(0, "## Metrics path")

sys.path.insert(0, "/kaggle/input/datasets/md0612/metrics")

from metrics.evaluator import Evaluator

metric_calculator = Evaluator()


def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]

    return torch.argmax(logits, dim=-1)


def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    predictions = tokenizer.batch_decode(
        predictions,
        skip_special_tokens=True,
    )

    labels = np.where(
        labels != -100,
        labels,
        tokenizer.pad_token_id,
    )

    golds = tokenizer.batch_decode(
        labels,
        skip_special_tokens=True,
    )

    results = metric_calculator.evaluate_on_all_sample(
        predictions,
        golds,
    )

    return {
        "ti_f1": results["TI"]["f1"],
        "tc_f1": results["TC"]["f1"],
        "ai_f1": results["AI"]["f1"],
        "ac_f1": results["AC"]["f1"],
        "avg_f1": results["avg_f1"],
    }

## Load Custom Trainer & Evaluator

Import custom GenerationTrainer and Evaluator from the project source.

- GenerationTrainer: Performs autoregressive generation during validation.
- Evaluator: Computes TI, TC, AI, AC and average F1.

In [ ]:
import sys

sys.path.insert(0, "/kaggle/input/datasets/md0612/ee-source")

from trainer.generation_trainer import GenerationTrainer

## 10. TrainingArguments

Các tham số training được giữ nguyên theo notebook gốc.

In [ ]:
training_args = TrainingArguments(
    output_dir="/kaggle/working/ee_qwen3/QLORA",

    num_train_epochs=10,

    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,

    eval_strategy="no",
    save_strategy="epoch",
    save_total_limit=15,

    load_best_model_at_end=False,

    gradient_checkpointing=True,
    report_to="none",
)


**Early Stop version**

In [ ]:
# training_args = TrainingArguments(
#     output_dir="/kaggle/working/ee_qwen3/QLORA",

#     num_train_epochs=10,

#     per_device_train_batch_size=1,
#     gradient_accumulation_steps=8,
#     learning_rate=2e-4,

#     eval_strategy="epoch",
#     save_strategy="epoch",
#     save_total_limit=15,

#     load_best_model_at_end=True,
#     metric_for_best_model="avg_f1",
#     greater_is_better=True,

#     gradient_checkpointing=True,
#     report_to="none",
# )

## 11. Giải phóng bộ nhớ và kiểm tra GPU

Dọn garbage collection/cache CUDA rồi in mức memory đang dùng trên từng GPU.

In [ ]:
gc.collect()
torch.cuda.empty_cache()
torch.cuda.ipc_collect()

for i in range(torch.cuda.device_count()):
    print(
        f"GPU {i}: "
        f"{torch.cuda.memory_allocated(i) / 1024**3:.2f} GB allocated, "
        f"{torch.cuda.memory_reserved(i) / 1024**3:.2f} GB reserved"
    )

## 12. Khởi tạo Trainer và kiểm tra batch

Khởi tạo `Trainer` với train/dev dataset và collator; lấy thử một batch từ evaluation dataloader để kiểm tra dữ liệu.

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    data_collator=ee_data_collator,
)

batch = next(iter(trainer.get_eval_dataloader()))

**Early Stop version**

In [ ]:
# trainer = GenerationTrainer(
#     model=model,
#     args=training_args,

#     train_dataset=train_dataset,

#     # Dùng cho validation loss
#     eval_dataset=dev_dataset,

#     # Dùng cho Generation evaluation
#     generation_dataset=dev_generation_dataset,

#     tokenizer=tokenizer,
#     evaluator=Evaluator(),

#     max_new_tokens=512,

#     data_collator=ee_data_collator,

#     callbacks=[
#         EarlyStoppingCallback(
#             early_stopping_patience=3,
#             early_stopping_threshold=0.001,
#         )
#     ],
# )

## 13. Chọn chế độ train: từ đầu hoặc resume

- Đặt `RESUME_FROM_CHECKPOINT = None` để train từ đầu.
- Đặt `RESUME_FROM_CHECKPOINT` bằng đường dẫn checkpoint để tiếp tục training.

Đường dẫn checkpoint hiện tại được giữ theo notebook gốc. Khi đổi model/dataset hoặc checkpoint, cập nhật các đường dẫn tương ứng ở những cell phía trên và cell này.

In [ ]:
# CHECKPOINT = "/kaggle/input/models/md0612/qwen3-4b-qlora-bkee-all-checkpoint/pytorch/default/4/checkpoint-8176"

# Train từ đầu: RESUME_FROM_CHECKPOINT = None
# Resume: RESUME_FROM_CHECKPOINT = CHECKPOINT
RESUME_FROM_CHECKPOINT = None

trainer.train(
    resume_from_checkpoint=RESUME_FROM_CHECKPOINT
)

## 14. Lưu model và tokenizer cuối cùng

Lưu adapter/model mà `Trainer` quản lý cùng tokenizer vào thư mục output.

In [ ]:
trainer.save_model("/kaggle/working/final")
tokenizer.save_pretrained("/kaggle/working/final")